In [ ]:
dbutils.widgets.text("ddl_paths", "")
ddl_paths = [path.strip() for path in dbutils.widgets.get("ddl_paths").split(",") if path.strip()]

In [ ]:
def split_statements(raw_sql):
    sql_without_comments = "\n".join(
        line for line in raw_sql.splitlines() if not line.strip().startswith("--")
    )
    return [stmt.strip() for stmt in sql_without_comments.split(";") if stmt.strip()]


for ddl_path in ddl_paths:
    with open(ddl_path) as f:
        statements = split_statements(f.read())

    applied, skipped, missing_principal = 0, 0, []
    for statement in statements:
        try:
            spark.sql(statement)
            applied += 1
        except Exception as e:
            if "already exists" in str(e).lower():
                skipped += 1
            elif "PRINCIPAL_DOES_NOT_EXIST" in str(e):
                missing_principal.append(statement)
            else:
                raise

    print(f"{applied} statements aplicados, {skipped} ja existentes (skip) -- {ddl_path}")
    for statement in missing_principal:
        print(f"AVISO: Grupo nao reconhecido pelo Unity Catalog, GRANT nao aplicado: {statement}")